In [1]:
import sys, os
# add ../.. to path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import control as ct
import numpy as np

import models.model_coef as TF
from models.mav_dynamics_control import MavDynamics
from models.compute_models import euler_state, quaternion_state
import parameters.control_parameters as AP
import parameters.aerosonde_parameters as MAV
import parameters.simulation_parameters as SIM
from message_types.msg_delta import MsgDelta

np.set_printoptions(suppress=True, precision=4, linewidth=120)


mav = MavDynamics(SIM.ts_simulation)
m = MAV.mass
g = MAV.gravity
u_star = AP.u_star
w_star = AP.w_star

x_trim = euler_state(TF.x_trim)
u_trim = TF.u_trim

def x_dot_euler(xdot_quat, x_euler, x_quat):
    dTe_dxq_of_xq = np.zeros((x_euler.size, x_quat.size))
    Te_out = x_euler
    n = x_quat.size
    eps = 0.001
    for i in range(n):
        x_quat_i = x_quat + eps * np.eye(n)[:, [i]]
        Te_outi = euler_state(x_quat_i)
        dTe_dxq_of_xq[:, [i]] = (Te_outi - Te_out) / eps
    xdot_euler = dTe_dxq_of_xq @ xdot_quat
    return xdot_euler

def uparms_update(t, x, u, params):
    x_euler = x.reshape((-1,1))
    x_quat = quaternion_state(x_euler)
    u = u.reshape((-1,1))
    delta = MsgDelta()
    delta.from_array(np.pad(u, ((0,2),(0,0)), constant_values=0))
    wind = np.zeros((6, 1))

    mav._state = x_quat  # x_quat is already the quaternion state
    mav._update_velocity_data(wind)
    forces_moments = mav._forces_moments(delta)
    xdot_quat = mav._f(x_quat, forces_moments)

    xdot_euler = x_dot_euler(xdot_quat, x_euler, x_quat)
    return xdot_euler.flatten()


def uparms_output(t, x, u, params):
    # Derive accelerations from dynamics so the output is a true function of (x, u),
    # not a time-history finite difference — required for correct ct.linearize Jacobians.
    xdot = uparms_update(t, x, u, params)
    u_body = x[3]
    v_body = x[4]
    w_body = x[5]
    udot   = xdot[3]
    vdot   = xdot[4]
    wdot   = xdot[5]
    theta  = x[7]
    V      = np.sqrt(u_body**2 + v_body**2 + w_body**2)
    Vgdot  = (u_body*udot + v_body*vdot + w_body*wdot) / V
    Edot   = Vgdot / MAV.gravity + np.sin(theta)
    Ldot   = Vgdot / MAV.gravity - np.sin(theta)
    return np.array([Edot, Ldot])


# Instantiate system
sys_nl = ct.NonlinearIOSystem(uparms_update, uparms_output, states=12, inputs=4, outputs=2)

# Linearize at trim
x_eq = x_trim.flatten()
u_eq = u_trim.flatten()
sys_lin = ct.linearize(sys_nl, x_eq, u_eq)

A = sys_lin.A
B = sys_lin.B
C = sys_lin.C
D = sys_lin.D

E1 = np.array([
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
])
E2 = np.array([
    [1, 0, 0, 0],
    [0, 0, 0, 1]
])
E3 = np.array([
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0,  0, 0, 0, 1, 0, 0, 0]
])
E4 = np.array([
    [0, 1, 0, 0],
    [0, 0, 1, 0]
])

A_lon = E1 @ A @ E1.T
B_lon = E1 @ B @ E2.T
C_lon = C @ E1.T
D_lon = D @ E2.T

A_lat = E3 @ A @ E3.T
B_lat = E3 @ B @ E4.T
C_lat = C @ E3.T
D_lat = D @ E4.T

print(f"A_lon =\n{A_lon}")
print(f"B_lon =\n{B_lon}")
print(f"C_lon =\n{C_lon}")
print(f"D_lon =\n{D_lon}")

A_lon =
[[-0.2822  0.4946 -1.2123 -9.7979  0.    ]
 [-0.5611 -4.4978 24.3714 -0.4874  0.    ]
 [ 0.1986 -3.993  -5.2947  0.      0.    ]
 [ 0.      0.      1.0001  0.      0.    ]
 [ 0.0497 -0.9988  0.     25.      0.    ]]
B_lon =
[[ -0.1392   9.4813]
 [ -2.5861   0.    ]
 [-36.1124   0.    ]
 [  0.       0.    ]
 [  0.       0.    ]]
C_lon =
[[-0.0316  0.0276 -0.     -0.0012  0.    ]
 [-0.0316  0.0276 -0.     -1.9988  0.    ]]
D_lon =
[[-0.0273  0.9653]
 [-0.0273  0.9653]]


In [2]:
import numpy as np
from numpy import concatenate, zeros, eye, diag

# Trim values for linearized Va output
u_star_lon = TF.x_trim.item(3)
w_star_lon = TF.x_trim.item(5)
Va_star    = TF.Va_trim

# Output matrices for altitude and airspeed from x_lon = [u, w, q, theta, h]
C_h   = np.array([[0, 0, 0, 0, 1]])                                       # h  = x_lon[4]
C_Va  = np.array([[u_star_lon/Va_star, w_star_lon/Va_star, 0, 0, 0]])     # Va ≈ linearized
D_h   = zeros((1, 2))
D_Va  = zeros((1, 2))
C_outer = concatenate([C_h, C_Va], axis=0)   # 2x5
D_outer = concatenate([D_h, D_Va], axis=0)   # 2x2

# Augmented state: [u, w, q, theta, h | Edot_int, Ldot_int | h_int, Va_int]
n_lon   = A_lon.shape[0]   # 5
n_tecs  = C_lon.shape[0]   # 2
n_outer = 2                # h_int, Va_int
n_aug   = n_lon + n_tecs + n_outer  # 9

Ahat = concatenate([
    concatenate([A_lon,   zeros((n_lon,   n_tecs + n_outer))], axis=1),
    concatenate([C_lon,   zeros((n_tecs,  n_tecs + n_outer))], axis=1),
    concatenate([C_outer, zeros((n_outer, n_tecs + n_outer))], axis=1),
], axis=0)

Bhat = concatenate([B_lon, D_lon, D_outer], axis=0)

print("Ahat ="); print(Ahat)
print("\nBhat ="); print(Bhat)

Ahat =
[[-0.2822  0.4946 -1.2123 -9.7979  0.      0.      0.      0.      0.    ]
 [-0.5611 -4.4978 24.3714 -0.4874  0.      0.      0.      0.      0.    ]
 [ 0.1986 -3.993  -5.2947  0.      0.      0.      0.      0.      0.    ]
 [ 0.      0.      1.0001  0.      0.      0.      0.      0.      0.    ]
 [ 0.0497 -0.9988  0.     25.      0.      0.      0.      0.      0.    ]
 [-0.0316  0.0276 -0.     -0.0012  0.      0.      0.      0.      0.    ]
 [-0.0316  0.0276 -0.     -1.9988  0.      0.      0.      0.      0.    ]
 [ 0.      0.      0.      0.      1.      0.      0.      0.      0.    ]
 [ 0.9988  0.0497  0.      0.      0.      0.      0.      0.      0.    ]]

Bhat =
[[ -0.1392   9.4813]
 [ -2.5861   0.    ]
 [-36.1124   0.    ]
 [  0.       0.    ]
 [  0.       0.    ]
 [ -0.0273   0.9653]
 [ -0.0273   0.9653]
 [  0.       0.    ]
 [  0.       0.    ]]


In [3]:
from numpy import diag
from scipy.linalg import solve_continuous_are, inv

#                  u      w      q      theta  h      Edot_i Ldot_i h_int  Va_int
Qlon = diag([    10.0,  100.0,  100.0,  10.0,  50.0,  60.0,  50.0,  80.0,  80.0])
Rlon = diag([1.0, 1.0])   # elevator, throttle

Plon = solve_continuous_are(Ahat, Bhat, Qlon, Rlon)
Klon = inv(Rlon) @ Bhat.T @ Plon

labels = ['u', 'w', 'q', 'theta', 'h', 'Edot_int', 'Ldot_int', 'h_int', 'Va_int']
print(f"Klon (de | dt) x [{', '.join(labels)}]")
print(Klon)

Klon (de | dt) x [u, w, q, theta, h, Edot_int, Ldot_int, h_int, Va_int]
[[   0.0284   -1.1954  -10.4264 -210.7265  -13.6977   -0.0151    0.0126   -8.5289    2.694 ]
 [   3.597    -1.4223    0.0806   35.7358    3.6718   -0.0477    0.0398    2.6941    8.5286]]
